# Auxiliary Datasets Integration

**Part 5 of the AQ Integration module.**

Parts 1 through 4 produced a single time × location backbone carrying
AQ targets (CAMS PM₂.₅), gridded meteorological context (ERA5), and
spatial context (Urban Atlas land cover, station registry).  This
notebook introduces the **auxiliary** predictors:

- **Traffic counters** - vehicle counts, mean speed, and heavy-vehicle
  share.
- **Station-collocated meteorology** - temperature, humidity, wind, and
  pressure measured at the AQ station itself, providing a more accurate
  point reading than the 0.25° ERA5 cell.
- **Static context** - elevation, land-use code, Local Climate Zone,
  distance to road, and energy intensity.

Upon completion of this notebook, you will be able to:

1. Convert any auxiliary feed into the project's canonical long-format
   schema by means of a *declarative contract* that specifies required
   columns, units, version, and source label.
2. Stack every source (AQ, satellite, reanalysis, auxiliary) into one
   integration-stage master table and pivot it to model-ready wide
   form.
3. Produce **predictor-readiness** and **coverage** reports that flag
   sparse features without silently discarding them.

The governing principle is that the integration stage prepares
structure; it does not select lags, transformations, train/test
splits, or features.  Such decisions belong to the modelling stage.

---

## Background: The Purpose of Auxiliary Integration

### Three Categories of Auxiliary Data

| Category | Examples | Time axis | Spatial axis |
|---|---|---|---|
| **Time-varying, per-station** | traffic counts, station meteo, energy demand | hourly / sub-hourly | one counter per station |
| **Time-varying, gridded** | reanalysis (already in Part 2) | hourly | full grid |
| **Static, per-station** | elevation, land cover, road distance, LCZ | none | constant per station |

### Dataset Contracts

A *contract* is the declarative agreement between an auxiliary feed
and the pipeline.  It specifies:

- the required columns that the adapter must produce,
- the mapping of value columns to canonical variable names,
- the units, which are recorded in metadata but not validated
  numerically,
- a version tag, which is carried into the output.

### Lag-Ready Structure, Not Lag-Applied

The wide model-ready table is sorted by ``(location_id, time)``.  This
arrangement allows lag features to be added downstream with a single
expression:

```python
wide["pm25_lag1h"] = wide.groupby("location_id")["pm2p5"].shift(1)
```

The integration stage terminates at this point.  Whether the model
employs lags of 1 hour, 3 hours, 24 hours, or none at all is a
decision reserved for the modelling notebook.

### Predictor Readiness

Before the master table is handed to a model, three questions must be
addressed:

- Which variables actually meet a coverage threshold?
- Which stations are feature-complete?
- Which predictors are too sparse to be used?

Sparse predictors are *flagged*, not silently discarded.  The
modelling stage may still elect to use them, either with imputation
or as auxiliary signals.

## Setup Procedure

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
_scripts = Path.cwd().parent.parent / "scripts" / "part5_aux"
if not _scripts.exists():
    _scripts = Path.cwd().parent / "scripts" / "part5_aux"
sys.path.insert(0, str(_scripts))

for prev in ("part3_temporal", "part4_spatial"):
    pp = _scripts.parent / prev
    if pp.exists():
        sys.path.insert(0, str(pp))

import aux_data_ingest as ingest
import aux_to_table as at
import join_master_table as jm
import coverage_report as cov

## Load the Station Registry

The notebook reuses the registry from Part 4 when it is available;
otherwise it falls back to the minimal ``stations_example.csv``
provided in Part 1.  Either source suffices to identify the stations
and to attach latitude and longitude to the auxiliary rows.

In [4]:
REG_PATH = Path("../../data/outputs/station_registry.csv")
BASE_PATH = Path("../../data/stations_example.csv")

if REG_PATH.exists():
    stations = pd.read_csv(REG_PATH)
    print(f"Loaded full registry from {REG_PATH} ({len(stations)} sites)")
else:
    stations = pd.read_csv(BASE_PATH)
    print(f"Loaded minimal stations from {BASE_PATH} ({len(stations)} sites)")
stations.head()

Loaded full registry from ..\..\data\outputs\station_registry.csv (5 sites)


,location_id,station_name,country,lat,lon,type,height_m,notes
0,ZAGREB01,Zagreb,HR,45.8150,15.9819,traffic,3.0,"City-centre site, partial urban canyon"
1,SPLIT01,Split,HR,43.5081,16.4402,background,2.0,NaN
2,RIJEKA01,Rijeka,HR,45.3271,14.4422,suburban,2.0,Coastal influence; sea breeze regime
3,OSIJEK01,Osijek,HR,45.5511,18.6939,background,2.0,Pannonian plain; agricultural surroundings
4,ZADAR01,Zadar,HR,44.1194,15.2314,background,2.0,NaN


## Generate Three Synthetic Auxiliary Datasets

Each call returns the adapter's *own* schema, which is identical in
form to what a real CSV ingestion routine would produce for the
corresponding feed.

In [5]:
rng = np.random.default_rng(42)

traffic_raw = ingest.synth_traffic(stations, rng=rng)
meteo_raw   = ingest.synth_station_meteo(stations, rng=rng)
context_raw = ingest.synth_static_context(stations, rng=rng)

print(f"traffic rows:  {len(traffic_raw):,}   ({traffic_raw.columns.tolist()})")
print(f"meteo   rows:  {len(meteo_raw):,}   ({meteo_raw.columns.tolist()})")
print(f"context rows:  {len(context_raw):,}      ({context_raw.columns.tolist()})")

traffic rows:  3,600   (['counter_id', 'time', 'vehicle_count', 'speed_kmh', 'heavy_fraction', 'location_id'])
meteo   rows:  3,600   (['location_id', 'time', 'temperature_c', 'humidity_pct', 'wind_speed_ms', 'wind_dir_deg', 'pressure_hpa'])
context rows:  5      (['location_id', 'elevation_m', 'land_use_code', 'lcz_class', 'road_dist_m', 'energy_intensity_kwh_m2'])


## Validate Contracts and Convert to Canonical Long Format

Each feed is wrapped in a ``DatasetContract`` that declares the
required columns, the mapping of values to canonical variable names,
the units, and a version tag.  The contract is validated *prior to*
conversion, so that a missing column produces an explicit failure at
the point of entry.

In [6]:
for raw, contract in [
    (traffic_raw, at.TRAFFIC_CONTRACT),
    (meteo_raw,   at.STATION_METEO_CONTRACT),
    (context_raw, at.STATIC_CONTEXT_CONTRACT),
]:
    rep = at.validate_contract(raw, contract)
    status = "OK" if rep.ok else "FAILED"
    print(f"  {contract.name}: {status}")
    for line in rep.issues:
        print(f"    - {line}")

  Hourly traffic counters: OK
  Station-colocated meteorology: OK
  Static per-station context: OK


In [7]:
traffic_long = at.to_canonical_long(
    traffic_raw, at.TRAFFIC_CONTRACT, stations, source_tz="Europe/Zagreb"
)
meteo_long = at.to_canonical_long(
    meteo_raw, at.STATION_METEO_CONTRACT, stations, source_tz="Europe/Zagreb"
)
context_long = at.to_canonical_long(
    context_raw, at.STATIC_CONTEXT_CONTRACT, stations
)

print(f"traffic_long: {len(traffic_long):,} rows, "
      f"variables: {sorted(traffic_long['variable'].unique())}")
print(f"meteo_long:   {len(meteo_long):,} rows, "
      f"variables: {sorted(meteo_long['variable'].unique())}")
print(f"context_long: {len(context_long):,} rows, "
      f"variables: {sorted(context_long['variable'].unique())}")
traffic_long.head()

traffic_long: 10,800 rows, variables: ['traffic_count_h', 'traffic_heavy_frac', 'traffic_speed_kmh']
meteo_long:   18,000 rows, variables: ['pressure_station', 'rh_station', 'temp_station', 'wind_dir_station', 'wind_station']
context_long: 25 rows, variables: ['elevation', 'energy_intensity', 'land_use_code', 'lcz_class', 'road_distance']


,time,location_id,lat,lon,variable,value,source,method,dataset_version
0,2024-05-31 22:00:00+00:00,ZAGREB01,45.815,15.9819,traffic_count_h,54.0,traffic_counters,colocated,synth-2024-06
1,2024-05-31 23:00:00+00:00,ZAGREB01,45.815,15.9819,traffic_count_h,7.0,traffic_counters,colocated,synth-2024-06
2,2024-06-01 00:00:00+00:00,ZAGREB01,45.815,15.9819,traffic_count_h,70.0,traffic_counters,colocated,synth-2024-06
3,2024-06-01 01:00:00+00:00,ZAGREB01,45.815,15.9819,traffic_count_h,77.0,traffic_counters,colocated,synth-2024-06
4,2024-06-01 02:00:00+00:00,ZAGREB01,45.815,15.9819,traffic_count_h,5.0,traffic_counters,colocated,synth-2024-06


## Incorporate the Outputs

In [9]:
ALIGNED_PATH = Path("../../data/outputs/aligned_long.csv")
MASTER_PATH  = Path("../../data/outputs/master_hourly_index.csv")

if ALIGNED_PATH.exists():
    aligned = pd.read_csv(ALIGNED_PATH, parse_dates=["time"])
    if aligned["time"].dt.tz is None:
        aligned["time"] = aligned["time"].dt.tz_localize("UTC")
    print(f"Loaded aligned long: {len(aligned):,} rows, "
          f"sources={sorted(aligned['source'].unique())}")
else:
    aligned = pd.DataFrame()
    print(f"No Part 3 aligned table at {ALIGNED_PATH} (skipped).")

if MASTER_PATH.exists():
    master = pd.read_csv(MASTER_PATH, parse_dates=["time"])
    if master["time"].dt.tz is None:
        master["time"] = master["time"].dt.tz_localize("UTC")
    print(f"Loaded master index: {len(master):,} cells")
else:
    times = pd.date_range("2024-06-01", "2024-06-30 23:00", freq="1h", tz="UTC")
    master = (
        stations[["location_id", "lat", "lon"]]
        .merge(pd.DataFrame({"time": times}), how="cross")
        .sort_values(["time", "location_id"])
        .reset_index(drop=True)
    )
    print(f"Built local master index: {len(master):,} cells")

Loaded aligned long: 32,390 rows, sources=['CAMS European AQ analysis', 'ground_station_synth', 'reanalysis', 'satellite']
Loaded master index: 3,600 cells


## Stack Every Source Into a Single Long Master Table

The ``stack_sources`` routine concatenates the canonical long
DataFrames and de-duplicates on
``(time, location_id, variable, source, method)``.  The first frame
takes precedence; therefore the inputs must be ordered from
most-trusted to least-trusted.

In [10]:
master_long = jm.stack_sources(
    aligned,        # AQ targets, satellite, reanalysis (from Parts 1-3)
    traffic_long,   # auxiliary: traffic counters
    meteo_long,     # auxiliary: station-collocated meteorology
    context_long,   # auxiliary: static context
)
print(f"Master long: {len(master_long):,} rows, "
      f"variables: {master_long['variable'].nunique()}, "
      f"sources: {master_long['source'].nunique()}")
master_long.head()

Master long: 61,215 rows, variables: 20, sources: 7


,time,location_id,lat,lon,variable,value,source,method,dataset_version
0,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,pressure_station,1009.275549,station_meteo,colocated,synth-2024-06
1,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,rh_station,69.910070,station_meteo,colocated,synth-2024-06
2,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,temp_station,16.247247,station_meteo,colocated,synth-2024-06
3,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,traffic_count_h,36.000000,traffic_counters,colocated,synth-2024-06
4,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,traffic_heavy_frac,0.129600,traffic_counters,colocated,synth-2024-06


## Pivot to Model-Ready Wide Form

The wide form contains one row per ``(time, location_id)`` and one
column per variable.  Static features are attached on ``location_id``,
so that they repeat across every timestamp at the corresponding
station.  The output is sorted by ``(location_id, time)``, which
places the addition of lag features downstream a single
``groupby.shift`` call away.  *The lags are not selected at this
stage.*

In [11]:
static_long = master_long[master_long["time"].isna()]
time_varying = master_long[master_long["time"].notna()]

wide = jm.to_model_ready(time_varying, static=static_long)
print(f"Wide table: {wide.shape}")
print("Columns:")
print("  index : ", [c for c in ("time", "location_id", "lat", "lon") if c in wide.columns])
print("  vars  : ", [c for c in wide.columns if c not in ("time", "location_id", "lat", "lon")])
wide.head()

Wide table: (3610, 24)
Columns:
  index :  ['time', 'location_id', 'lat', 'lon']
  vars  :  ['blh', 'd2m', 'pm2p5', 'pressure_station', 'rh_station', 'sp', 't2m', 'temp_station', 'tp', 'traffic_count_h', 'traffic_heavy_frac', 'traffic_speed_kmh', 'wind_dir_station', 'wind_speed_10m', 'wind_station', 'elevation', 'energy_intensity', 'land_use_code', 'lcz_class', 'road_distance']


,time,location_id,lat,lon,blh,d2m,pm2p5,pressure_station,rh_station,sp,...,traffic_heavy_frac,traffic_speed_kmh,wind_dir_station,wind_speed_10m,wind_station,elevation,energy_intensity,land_use_code,lcz_class,road_distance
0,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,NaN,NaN,NaN,1009.275549,69.910070,NaN,...,0.129600,66.100209,107.573548,NaN,2.436665,103.9,83.3,11100.0,8.0,230.8
1,2024-05-31 23:00:00+00:00,OSIJEK01,45.5511,18.6939,NaN,NaN,NaN,1009.275475,66.560437,NaN,...,0.152308,66.416838,247.379308,NaN,4.206739,103.9,83.3,11100.0,8.0,230.8
2,2024-06-01 00:00:00+00:00,OSIJEK01,45.5511,18.6939,53.547312,287.945847,4.511617,1017.653173,58.632331,99849.159319,...,0.113394,65.332589,81.954993,0.874448,2.495647,103.9,83.3,11100.0,8.0,230.8
3,2024-06-01 01:00:00+00:00,OSIJEK01,45.5511,18.6939,41.588441,287.400710,4.424342,1012.569695,75.668053,99853.607286,...,0.096405,61.020767,150.169455,1.134361,4.061128,103.9,83.3,11100.0,8.0,230.8
4,2024-06-01 02:00:00+00:00,OSIJEK01,45.5511,18.6939,50.814823,287.391405,4.297293,1018.308746,62.777131,99857.365486,...,0.122038,60.466870,26.329890,1.404790,3.530075,103.9,83.3,11100.0,8.0,230.8


## Coverage and Predictor-Readiness Report

Before the master table is declared ready, three questions must be
addressed:

1. **By (source, variable)** - what fraction of the master cells does
   each predictor actually fill?
2. **By station × variable** - are some stations feature-complete
   while others remain sparse?
3. **Verdict per variable** - ready, borderline, or not_ready,
   evaluated against configurable thresholds.

These are integration-stage reports.  A variable that falls below the
readiness threshold is flagged, not silently discarded; the modelling
stage may still elect to use it.

In [12]:
report = cov.summarize(
    master_long,
    master=master,
    thresholds=cov.ReadinessThresholds(ready_min=0.85, borderline_min=0.50),
)
cov.print_summary(report)


=== BY_SOURCE_VARIABLE ===
                   source           variable  observed  expected  coverage
     ground_station_synth              pm2p5      3583      3600     0.995
CAMS European AQ analysis              pm2p5      3600      3600     1.000
               reanalysis                blh      3600      3600     1.000
               reanalysis                d2m      3600      3600     1.000
               reanalysis                 sp      3600      3600     1.000
               reanalysis                t2m      3600      3600     1.000
               reanalysis                 tp      3600      3600     1.000
               reanalysis     wind_speed_10m      3600      3600     1.000
                satellite              pm2p5      3600      3600     1.000
            station_meteo   pressure_station      3600      3600     1.000
            station_meteo         rh_station      3600      3600     1.000
            station_meteo       temp_station      3600      3600     1.0

## Save the Integrated Master Dataset

Three artefacts are persisted so that the modelling stage may consume
the pipeline output directly:

- ``master_long.csv``       — every source stacked in canonical long format.
- ``master_wide.csv``       — the model-ready wide table.

In [13]:
out_dir = Path("../../data/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

master_long_path = out_dir / "master_long.csv"
master_wide_path = out_dir / "master_wide.csv"
master_long.to_csv(master_long_path, index=False)
wide.to_csv(master_wide_path, index=False)
print(f"Wrote {master_long_path}  ({len(master_long):,} rows)")
print(f"Wrote {master_wide_path}  ({wide.shape[0]:,} x {wide.shape[1]} cols)")

Wrote ..\..\data\outputs\master_long.csv  (61,215 rows)
Wrote ..\..\data\outputs\master_wide.csv  (3,610 x 24 cols)


## Optional Preview: Adding Lag Features

This section serves only to demonstrate that the wide table is
structured correctly.  In practice, the modelling notebook performs
this operation, not the integration stage.  Two pollutant or
meteorological columns are selected, and lags of 1 hour, 3 hours, and
24 hours are added, grouped by station.

In [14]:
lag_candidates = [c for c in (
    "pm2p5", "pm2p5_conc", "t2m", "temp_station",
    "traffic_count_h", "wind_station",
) if c in wide.columns]

if lag_candidates:
    sample = jm.add_lag_features(
        wide, columns=lag_candidates[:2], lags_hours=(1, 3, 24)
    )
    show_cols = ["time", "location_id"] + lag_candidates[:2] + [
        f"{c}_lag1h" for c in lag_candidates[:2]
    ]
    sample[show_cols].head(8)
else:
    print("No suitable columns found for the lag demo.")

## Summary

The integration-stage **master dataset** has now been constructed.
This notebook accomplished the following:

1. Generated three realistic auxiliary feeds (traffic, station
   meteorology, and static context) by means of the adapter helpers.
2. Wrapped each feed in a **dataset contract** and validated it prior
   to conversion.
3. Converted every feed into the canonical long-format schema, with
   units and version tags preserved.
4. Incorporated the outputs of Parts 3 and 4 where available.
5. Stacked every source into a single long master table, and then
   pivoted it to a model-ready wide table sorted in lag-ready order.
6. Produced coverage and predictor-readiness reports.
7. Persisted ``master_long.csv``, ``master_wide.csv``, and a metadata
   sidecar.